# Sensitivity Until First False Positive - All Encodings and K-sizes

This notebook computes sensitivity until first false positive using MULTIPLE METRICS for all encodings (hp, dayhoff, protein) and ksizes. 

We analyze binary classifications:
- Same family: True/False
- Same superfamily: True/False
- Same fold: True/False
- Same class: True/False

For each combination of metric, encoding, ksize, and SCOP level.

## Setup and Imports

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300

## Define Data Files and Metrics

Using CSV files from: `~/data/scope/results-2025-12-25-average_kmer_rarity/`

In [6]:
data_dir = Path("/Users/olga/data/scope/results-2025-12-25-average_kmer_rarity")

# Define all CSV files
csv_files = {
    # Dayhoff encodings
    'dayhoff_k10': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k10.scaled1.kmerseek.results.csv',
    'dayhoff_k11': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k11.scaled1.kmerseek.results.csv',
    'dayhoff_k12': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k12.scaled1.kmerseek.results.csv',
    'dayhoff_k13': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k13.scaled1.kmerseek.results.csv',
    'dayhoff_k14': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k14.scaled1.kmerseek.results.csv',
    'dayhoff_k15': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k15.scaled1.kmerseek.results.csv',
    
    # HP encodings
    'hp_k15': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k15.scaled1.kmerseek.results.csv',
    'hp_k16': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k16.scaled1.kmerseek.results.csv',
    'hp_k17': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k17.scaled1.kmerseek.results.csv',
    'hp_k18': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k18.scaled1.kmerseek.results.csv',
    'hp_k19': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k19.scaled1.kmerseek.results.csv',
    'hp_k20': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k20.scaled1.kmerseek.results.csv',
    
    # Protein encodings
    'protein_k5': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.protein.k5.scaled1.kmerseek.results.csv',
    'protein_k6': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.protein.k6.scaled1.kmerseek.results.csv',
    'protein_k7': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.protein.k7.scaled1.kmerseek.results.csv',
    'protein_k8': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.protein.k8.scaled1.kmerseek.results.csv',
    'protein_k9': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.protein.k9.scaled1.kmerseek.results.csv',
    'protein_k10': 'astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.protein.k10.scaled1.kmerseek.results.csv',
}

# Max rows to read for large files (15M rows)
MAX_ROWS = 15_000_000

# Define all metrics to test
SCORE_METRICS = [
    # Key metrics
    'average_kmer_rarity',
    'observed_over_expected',
    'prob_random_cooccurrence',
    'prob_random_cooccurrence_symmetric',
    'tfidf',
    
    # Basic similarity metrics
    'containment',
    'max_containment',
    'jaccard',
    'n_intersecting_hashes',
    'containment_target_in_query',
    'f_weighted_target_in_query',
    
    # Database-aware metrics
    'average_database_kmer_frequency',
    'sum_database_frequencies_of_matches',
    'expected_intersecting_hashes',
]

# SCOP levels to analyze
SCOP_LEVELS = ['family', 'superfamily', 'fold', 'class']

print(f"Total files: {len(csv_files)}")
print(f"Max rows per file: {MAX_ROWS:,}")
print(f"Metrics to evaluate: {len(SCORE_METRICS)}")
print(f"SCOP levels: {SCOP_LEVELS}")
print("\nFiles per encoding:")
for encoding in ['dayhoff', 'hp', 'protein']:
    count = sum(1 for k in csv_files.keys() if k.startswith(encoding))
    print(f"  {encoding}: {count}")

Total files: 18
Max rows per file: 15,000,000
Metrics to evaluate: 14
SCOP levels: ['family', 'superfamily', 'fold', 'class']

Files per encoding:
  dayhoff: 6
  hp: 6
  protein: 6


## Function: Sensitivity Until First False Positive (Multiple Metrics)

Based on code from notebook 53. This function:
1. Sorts matches by a given score metric (descending for most, ascending for p-values)
2. Finds the first false positive match
3. Calculates sensitivity = # true positives before first FP / total possible true positives

In [ ]:
def compute_sensitivity_for_metric(df, score_metric, scop_level='family'):
    """
    Compute sensitivity until first false positive for a given score metric and SCOP level.
    
    Based on compute_sensitivity_curve from notebook 53.
    
    Parameters:
    -----------
    df : polars.DataFrame
        Must contain: query_name, target_name, query_md5, target_md5, score_metric columns
    score_metric : str
        Column name to rank by (e.g., 'tfidf', 'average_kmer_rarity')
    scop_level : str
        One of: 'family', 'superfamily', 'fold', 'class'
    
    Returns:
    --------
    List of sensitivity values (one per query)
    """
    # Parse SCOP lineages from name field
    df = df.with_columns([
        pl.col("query_name").str.split(" ").list.get(1).alias("query_lineage"),
        pl.col("target_name").str.split(" ").list.get(1).alias("target_lineage"),
    ])
    
    # Extract SCOP level
    if scop_level == "family":
        df = df.with_columns([
            pl.col("query_lineage").alias("query_scop"),
            pl.col("target_lineage").alias("target_scop"),
        ])
    elif scop_level == "superfamily":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1) + pl.lit(".") + parts_q.list.get(2)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1) + pl.lit(".") + parts_t.list.get(2)).alias("target_scop"),
        ])
    elif scop_level == "fold":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1)).alias("target_scop"),
        ])
    elif scop_level == "class":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            parts_q.list.get(0).alias("query_scop"),
            parts_t.list.get(0).alias("target_scop"),
        ])
    
    df = df.with_columns([
        (pl.col("query_scop") == pl.col("target_scop")).alias("same_scop")
    ])
    
    # Remove self-hits
    df = df.filter(pl.col("query_md5") != pl.col("target_md5"))
    
    # Get unique queries
    queries = df["query_name"].unique().to_list()
    
    sensitivities = []
    for query in queries:
        qdf = df.filter(pl.col("query_name") == query)
        
        # Determine sort order: most metrics want descending (higher is better)
        # but p-values want ascending (lower is better)
        descending = True
        if 'prob' in score_metric.lower() or 'p_value' in score_metric.lower():
            descending = False
        
        qdf = qdf.sort(score_metric, descending=descending, nulls_last=True)
        
        same_vals = qdf["same_scop"].to_list()
        n_positives = sum(same_vals)
        
        if n_positives == 0:
            # No true positives for this query
            sensitivities.append(0.0)
            continue
        
        # Find first false positive
        first_fp = next((i for i, v in enumerate(same_vals) if not v), None)
        
        if first_fp is None:
            # No false positives - retrieved all positives
            sensitivity = 1.0
        elif first_fp == 0:
            # First hit is FP
            sensitivity = 0.0
        else:
            # Sensitivity = fraction of positives retrieved before first FP
            sensitivity = min(first_fp / n_positives, 1.0)
        
        sensitivities.append(sensitivity)
    
    return sensitivities


def calculate_all_sensitivities(df, encoding, ksize):
    """
    Calculate sensitivity for all metrics and all SCOP levels.
    
    Returns a DataFrame with one row per query/metric/scop_level combination.
    """
    results = []
    
    # Get unique queries for this file
    queries = df["query_name"].unique().to_list()
    
    for metric in tqdm(SCORE_METRICS, desc=f"Metrics ({encoding} k{ksize})", leave=False):
        # Skip if metric doesn't exist in this file
        if metric not in df.columns:
            print(f"  Warning: {metric} not in columns, skipping")
            continue
        
        for scop_level in SCOP_LEVELS:
            try:
                sensitivities = compute_sensitivity_for_metric(df, metric, scop_level)
                
                # Create rows for this metric/level combination
                for query, sens in zip(queries, sensitivities):
                    results.append({
                        'query_name': query,
                        'encoding': encoding,
                        'ksize': ksize,
                        'metric': metric,
                        'scop_level': scop_level,
                        'sensitivity': sens
                    })
            except Exception as e:
                print(f"  Error with {metric} / {scop_level}: {e}")
                continue
    
    return pl.DataFrame(results)

: 

## Process All Files

Load each CSV (all columns, limited to 15M rows for large files) and compute sensitivity metrics for all metrics and SCOP levels.

In [ ]:
# Process all files
all_sensitivity_results = []

for key, filename in tqdm(csv_files.items(), desc="Processing files"):
    # Parse encoding and ksize from key
    parts = key.split('_')
    encoding = parts[0]
    ksize = int(parts[1].replace('k', ''))
    
    filepath = data_dir / filename
    
    print(f"\n{'='*80}")
    print(f"Processing {encoding} k={ksize}")
    print(f"File: {filename}")
    print(f"Size: {filepath.stat().st_size / 1e9:.2f} GB")
    
    try:
        # Read CSV with polars - READ ALL COLUMNS, limit rows for large files
        file_size_gb = filepath.stat().st_size / 1e9
        
        if file_size_gb > 8.0:  # Large file
            print(f"  Large file detected, reading first {MAX_ROWS:,} rows")
            df = pl.read_csv(filepath, n_rows=MAX_ROWS)
        else:
            print(f"  Reading all rows")
            df = pl.read_csv(filepath)
        
        print(f"  Loaded {len(df):,} rows, {len(df.columns)} columns")
        print(f"  Columns: {', '.join(df.columns[:10])}...")
        
        # Calculate sensitivity metrics for all metrics and SCOP levels
        sensitivity_df = calculate_all_sensitivities(df, encoding, ksize)
        
        all_sensitivity_results.append(sensitivity_df)
        
        print(f"  ✓ Computed {len(sensitivity_df):,} sensitivity measurements")
        print(f"    ({len(df['query_name'].unique())} queries × {len(SCORE_METRICS)} metrics × {len(SCOP_LEVELS)} levels)")
        
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        continue

# Combine all results
print("\n" + "="*80)
print("Combining all results...")
sensitivity_all = pl.concat(all_sensitivity_results)

print(f"\nTotal sensitivity measurements: {len(sensitivity_all):,}")
print(f"Shape: {sensitivity_all.shape}")
print(f"\nUnique values:")
print(f"  Encodings: {sensitivity_all['encoding'].n_unique()}")
print(f"  K-sizes: {sorted(sensitivity_all['ksize'].unique().to_list())}")
print(f"  Metrics: {sensitivity_all['metric'].n_unique()}")
print(f"  SCOP levels: {sensitivity_all['scop_level'].n_unique()}")

Processing files:   0%|          | 0/18 [00:00<?, ?it/s]


Processing dayhoff k=10
File: astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k10.scaled1.kmerseek.results.csv
Size: 1.35 GB
  Reading all rows
  Loaded 2,584,617 rows, 30 columns
  Columns: query_name, query_md5, target_name, target_md5, containment, n_intersecting_hashes, ksize, scaled, moltype, jaccard...


Processing files:   6%|▌         | 1/18 [34:15<9:42:18, 2055.20s/it]

  ✓ Computed 662,244 sensitivity measurements
    (15051 queries × 14 metrics × 4 levels)

Processing dayhoff k=11
File: astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.dayhoff.k11.scaled1.kmerseek.results.csv
Size: 0.33 GB
  Reading all rows
  Loaded 629,133 rows, 30 columns
  Columns: query_name, query_md5, target_name, target_md5, containment, n_intersecting_hashes, ksize, scaled, moltype, jaccard...


Processing files:  11%|█         | 2/18 [1:21:36<11:11:23, 2517.71s/it]

...

  ✓ Computed 667,744 sensitivity measurements
    (15176 queries × 14 metrics × 4 levels)

Processing hp k=16
File: astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k16.scaled1.kmerseek.results.csv
Size: 30.99 GB
  Large file detected, reading first 15,000,000 rows
  Loaded 15,000,000 rows, 30 columns
  Columns: query_name, query_md5, target_name, target_md5, containment, n_intersecting_hashes, ksize, scaled, moltype, jaccard...


Processing files:  44%|████▍     | 8/18 [4:38:48<7:05:09, 2550.92s/it]

  ✓ Computed 667,744 sensitivity measurements
    (15176 queries × 14 metrics × 4 levels)

Processing hp k=17
File: astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k17.scaled1.kmerseek.results.csv
Size: 15.50 GB
  Large file detected, reading first 15,000,000 rows
  Loaded 15,000,000 rows, 30 columns
  Columns: query_name, query_md5, target_name, target_md5, containment, n_intersecting_hashes, ksize, scaled, moltype, jaccard...


Processing files:  50%|█████     | 9/18 [5:43:32<7:25:09, 2967.71s/it]

  ✓ Computed 667,744 sensitivity measurements
    (15176 queries × 14 metrics × 4 levels)

Processing hp k=18
File: astral-scopedom-seqres-gd-sel-gs-bib-40-2.08.fa.hp.k18.scaled1.kmerseek.results.csv
Size: 7.77 GB
  Reading all rows
  Loaded 14,306,424 rows, 30 columns
  Columns: query_name, query_md5, target_name, target_md5, containment, n_intersecting_hashes, ksize, scaled, moltype, jaccard...


In [ ]:
# Display sample of results
sensitivity_all.head(20)

## AUROC Calculation

Calculate Area Under ROC Curve for each metric, encoding, ksize, and SCOP level (family, superfamily, fold).

AUROC measures the ability to rank true positives higher than false positives across all thresholds.

In [ ]:
def compute_auroc_for_metric(df, score_metric, scop_level='family'):
    """
    Compute AUROC for a given score metric and SCOP level.
    
    Parameters:
    -----------
    df : polars.DataFrame
        Must contain: query_name, target_name, query_md5, target_md5, score_metric columns
    score_metric : str
        Column name to use for scoring
    scop_level : str
        One of: 'family', 'superfamily', 'fold'
    
    Returns:
    --------
    float: AUROC score (or None if cannot compute)
    """
    # Parse SCOP lineages
    df = df.with_columns([
        pl.col("query_name").str.split(" ").list.get(1).alias("query_lineage"),
        pl.col("target_name").str.split(" ").list.get(1).alias("target_lineage"),
    ])
    
    # Extract SCOP level
    if scop_level == "family":
        df = df.with_columns([
            pl.col("query_lineage").alias("query_scop"),
            pl.col("target_lineage").alias("target_scop"),
        ])
    elif scop_level == "superfamily":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1) + pl.lit(".") + parts_q.list.get(2)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1) + pl.lit(".") + parts_t.list.get(2)).alias("target_scop"),
        ])
    elif scop_level == "fold":
        parts_q = pl.col("query_lineage").str.split(".")
        parts_t = pl.col("target_lineage").str.split(".")
        df = df.with_columns([
            (parts_q.list.get(0) + pl.lit(".") + parts_q.list.get(1)).alias("query_scop"),
            (parts_t.list.get(0) + pl.lit(".") + parts_t.list.get(1)).alias("target_scop"),
        ])
    
    df = df.with_columns([
        (pl.col("query_scop") == pl.col("target_scop")).alias("same_scop")
    ])
    
    # Remove self-hits and nulls
    df = df.filter(pl.col("query_md5") != pl.col("target_md5"))
    df = df.filter(pl.col(score_metric).is_not_null())
    
    if len(df) == 0:
        return None
    
    y_true = df["same_scop"].to_numpy().astype(int)
    y_score = df[score_metric].to_numpy()
    
    # Check if we have both classes
    if len(np.unique(y_true)) < 2:
        return None
    
    # For probability metrics (lower is better), negate scores
    if 'prob' in score_metric.lower():
        y_score = -y_score
    
    try:
        auroc = roc_auc_score(y_true, y_score)
        return auroc
    except:
        return None

In [ ]:
# Calculate AUROC for all files
all_auroc_results = []

for key, filename in tqdm(csv_files.items(), desc="Computing AUROC"):
    parts = key.split('_')
    encoding = parts[0]
    ksize = int(parts[1].replace('k', ''))
    filepath = data_dir / filename
    
    print(f"\n{'='*80}")
    print(f"AUROC: {encoding} k={ksize}")
    
    try:
        file_size_gb = filepath.stat().st_size / 1e9
        if file_size_gb > 2.0:
            df = pl.read_csv(filepath, n_rows=MAX_ROWS)
        else:
            df = pl.read_csv(filepath)
        
        print(f"  {len(df):,} rows")
        
        for metric in tqdm(SCORE_METRICS, desc="  Metrics", leave=False):
            if metric not in df.columns:
                continue
            
            for scop_level in ['family', 'superfamily', 'fold']:
                try:
                    auroc = compute_auroc_for_metric(df, metric, scop_level)
                    if auroc is not None:
                        all_auroc_results.append({
                            'encoding': encoding,
                            'ksize': ksize,
                            'metric': metric,
                            'scop_level': scop_level,
                            'auroc': auroc
                        })
                except Exception as e:
                    pass
        
        print(f"  ✓ Done")
    except Exception as e:
        print(f"  ✗ ERROR: {e}")

auroc_df = pl.DataFrame(all_auroc_results)
print(f"\nTotal AUROC measurements: {len(auroc_df):,}")

NameError: name 'tqdm' is not defined

In [ ]:
auroc_df.head(20)

NameError: name 'auroc_df' is not defined

## Top AUROC Performers

In [ ]:
for level in ['family', 'superfamily', 'fold']:
    print(f"\n{'='*80}")
    print(f"Top 10 AUROC - {level.upper()}")
    print(f"{'='*80}")
    top = auroc_df.filter(pl.col('scop_level') == level).sort('auroc', descending=True).head(10)
    print(top.to_pandas().to_string(index=False))

## AUROC Heatmaps by SCOP Level

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for idx, level in enumerate(['family', 'superfamily', 'fold']):
    ax = axes[idx]
    level_data = auroc_df.filter(pl.col('scop_level') == level)
    level_data = level_data.with_columns(
        (pl.col('encoding') + '_k' + pl.col('ksize').cast(str)).alias('enc_k')
    )
    level_pd = level_data.to_pandas()
    pivot = level_pd.pivot_table(values='auroc', index='metric', columns='enc_k')
    pivot['mean'] = pivot.mean(axis=1)
    pivot = pivot.sort_values('mean', ascending=False).drop('mean', axis=1)
    
    sns.heatmap(pivot, annot=False, cmap='RdYlGn', vmin=0.5, vmax=1.0, 
                ax=ax, cbar_kws={'label': 'AUROC'})
    ax.set_title(f'{level.capitalize()}', fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.suptitle('AUROC by Metric, Encoding, K-size', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('auroc_heatmaps.pdf', bbox_inches='tight')
plt.show()

In [ ]:
auroc_df.write_csv('auroc_all_metrics.csv')
print('✓ Saved AUROC results to: auroc_all_metrics.csv')

## Summary Statistics by Metric, Encoding, K-size, and SCOP Level

In [ ]:
# Calculate summary statistics
summary_stats = sensitivity_all.group_by(['metric', 'encoding', 'ksize', 'scop_level']).agg([
    pl.col('sensitivity').mean().alias('mean_sensitivity'),
    pl.col('sensitivity').median().alias('median_sensitivity'),
    pl.col('sensitivity').std().alias('std_sensitivity'),
    pl.col('sensitivity').quantile(0.25).alias('q25_sensitivity'),
    pl.col('sensitivity').quantile(0.75).alias('q75_sensitivity'),
    pl.col('sensitivity').count().alias('n_queries')
]).sort(['scop_level', 'metric', 'encoding', 'ksize'])

print(f"Summary statistics shape: {summary_stats.shape}")
summary_stats.head(20)

## Top Performing Metrics by SCOP Level

In [ ]:
# Find best metric/encoding/ksize for each SCOP level
for level in SCOP_LEVELS:
    print(f"\n{'='*80}")
    print(f"Top 10 configurations for {level.upper()} level")
    print(f"{'='*80}")
    
    level_stats = summary_stats.filter(pl.col('scop_level') == level).sort(
        'mean_sensitivity', descending=True
    ).head(10)
    
    print(level_stats.select([
        'metric', 'encoding', 'ksize', 'mean_sensitivity', 'median_sensitivity', 'n_queries'
    ]).to_pandas().to_string(index=False))

## Boxplots: Sensitivity by SCOP Level and Metric

Show distribution of sensitivity values for all metrics across encodings/ksizes.

In [ ]:
# Convert to pandas for seaborn plotting
sensitivity_pd = sensitivity_all.to_pandas()

# Create ordered categories
sensitivity_pd['scop_level'] = pd.Categorical(
    sensitivity_pd['scop_level'],
    categories=SCOP_LEVELS,
    ordered=True
)

# Add combined label
sensitivity_pd['encoding_ksize'] = sensitivity_pd['encoding'] + ' k=' + sensitivity_pd['ksize'].astype(str)

In [ ]:
# Create boxplots for TOP 6 METRICS at each SCOP level
# First, identify top 6 metrics overall
top_metrics = summary_stats.group_by('metric').agg(
    pl.col('mean_sensitivity').mean().alias('overall_mean')
).sort('overall_mean', descending=True).head(6)['metric'].to_list()

print(f"Top 6 metrics overall: {top_metrics}")

# Filter to top metrics
sensitivity_top_pd = sensitivity_pd[sensitivity_pd['metric'].isin(top_metrics)].copy()

# Create figure
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
axes = axes.flatten()

colors = {'hp': '#1f77b4', 'dayhoff': '#ff7f0e', 'protein': '#2ca02c'}

for idx, level in enumerate(SCOP_LEVELS):
    ax = axes[idx]
    
    # Filter data for this SCOP level
    data = sensitivity_top_pd[sensitivity_top_pd['scop_level'] == level]
    
    # Create boxplot
    sns.boxplot(
        data=data,
        x='metric',
        y='sensitivity',
        hue='encoding',
        palette=colors,
        ax=ax,
        showfliers=False
    )
    
    ax.set_title(f'{level.capitalize()} Level', fontsize=16, fontweight='bold')
    ax.set_xlabel('Metric', fontsize=13)
    ax.set_ylabel('Sensitivity at First FP', fontsize=13)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=45, labelsize=10)
    ax.legend(title='Encoding', fontsize=10)

plt.tight_layout()
plt.suptitle('Sensitivity Until First False Positive - Top 6 Metrics\n' +
             'All Encodings and K-sizes', 
             fontsize=18, fontweight='bold', y=1.02)

plt.savefig('sensitivity_boxplots_top_metrics.pdf', bbox_inches='tight')
plt.show()

## Heatmap: Best Metric Performance by Encoding and K-size

In [ ]:
# For each encoding/ksize/scop_level, find the best metric
best_metrics = summary_stats.sort('mean_sensitivity', descending=True).group_by(
    ['encoding', 'ksize', 'scop_level']
).first()

# Create heatmaps
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
axes = axes.flatten()

for idx, level in enumerate(SCOP_LEVELS):
    ax = axes[idx]
    
    # Pivot data for heatmap
    level_data = best_metrics.filter(pl.col('scop_level') == level)
    
    # Create encoding_ksize column for x-axis
    level_data = level_data.with_columns(
        (pl.col('encoding') + '_k' + pl.col('ksize').cast(str)).alias('enc_k')
    )
    
    # Convert to pandas and create pivot
    level_pd = level_data.to_pandas()
    
    # Create a matrix: rows = metrics, cols = encoding_ksize
    pivot_data = level_pd.pivot_table(
        values='mean_sensitivity',
        index='metric',
        columns='enc_k',
        aggfunc='first'
    )
    
    # Create heatmap
    sns.heatmap(
        pivot_data,
        annot=False,
        cmap='RdYlGn',
        vmin=0,
        vmax=1,
        ax=ax,
        cbar_kws={'label': 'Mean Sensitivity'},
        xticklabels=True,
        yticklabels=True
    )
    
    ax.set_title(f'{level.capitalize()} Level - Best Metric Performance', 
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Encoding_Ksize', fontsize=12)
    ax.set_ylabel('Metric', fontsize=12)
    ax.tick_params(axis='x', rotation=90, labelsize=8)
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.suptitle('Best Metric Performance Heatmaps by SCOP Level', 
             fontsize=16, fontweight='bold', y=1.01)

plt.savefig('sensitivity_heatmaps_best_metrics.pdf', bbox_inches='tight')
plt.show()

## Metric Comparison: Which metrics perform best overall?

In [ ]:
# Average performance across all encodings, ksizes, and SCOP levels
metric_overall = summary_stats.group_by('metric').agg([
    pl.col('mean_sensitivity').mean().alias('overall_mean'),
    pl.col('mean_sensitivity').std().alias('overall_std'),
    pl.col('mean_sensitivity').max().alias('best_case'),
    pl.col('mean_sensitivity').min().alias('worst_case')
]).sort('overall_mean', descending=True)

print("\n" + "="*80)
print("OVERALL METRIC PERFORMANCE RANKING")
print("="*80)
print(metric_overall.to_pandas().to_string(index=False))

In [ ]:
# Visualize overall metric performance
fig, ax = plt.subplots(figsize=(12, 8))

metric_overall_pd = metric_overall.to_pandas().sort_values('overall_mean', ascending=True)

y_pos = np.arange(len(metric_overall_pd))
ax.barh(y_pos, metric_overall_pd['overall_mean'], 
        xerr=metric_overall_pd['overall_std'],
        alpha=0.8, color='steelblue', capsize=5)

ax.set_yticks(y_pos)
ax.set_yticklabels(metric_overall_pd['metric'], fontsize=11)
ax.set_xlabel('Mean Sensitivity (± std)', fontsize=13, fontweight='bold')
ax.set_title('Overall Metric Performance\nAveraged across all encodings, k-sizes, and SCOP levels',
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('metric_ranking_overall.pdf', bbox_inches='tight')
plt.show()

## Save Results

In [ ]:
# Save full results
output_file = 'sensitivity_all_metrics_encodings_ksizes.parquet'
sensitivity_all.write_parquet(output_file)
print(f"✓ Saved full results to: {output_file}")
print(f"  Size: {Path(output_file).stat().st_size / 1e6:.1f} MB")

# Save summary statistics
summary_file = 'sensitivity_summary_stats_all_metrics.csv'
summary_stats.write_csv(summary_file)
print(f"✓ Saved summary statistics to: {summary_file}")

# Save metric rankings
ranking_file = 'metric_performance_ranking.csv'
metric_overall.write_csv(ranking_file)
print(f"✓ Saved metric rankings to: {ranking_file}")